In [1]:
import pandas as pd
import os

current_dir = os.path.abspath('.')
root_dir = os.path.dirname(current_dir) 
file_path = os.path.join(root_dir, 'data/raw', 'default of credit card clients.csv')
    
# Charger le CSV dans un DataFrame
df = pd.read_csv(file_path)

## Nettoyage suite à Audit

In [2]:
# Afficher les répartitions avant nettoyage
print("\n=== RÉPARTITION AVANT NETTOYAGE ===")
print("MARRIAGE :")
marriage_counts = df['MARRIAGE'].value_counts().sort_index()
for value, count in marriage_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

print("\nEDUCATION :")
education_counts = df['EDUCATION'].value_counts().sort_index()
for value, count in education_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

# Nettoyage de la colonne MARRIAGE
print("=== NETTOYAGE DE LA COLONNE MARRIAGE ===")
print(f"Valeurs originales dans MARRIAGE : {df['MARRIAGE'].unique().tolist()}")

# Remplacer les valeurs en dehors de [1, 2, 3] par 2 (autres)
df['MARRIAGE'] = df['MARRIAGE'].replace([x for x in df['MARRIAGE'].unique() if x not in [1, 2, 3]], 3)

print(f"Valeurs après nettoyage : {df['MARRIAGE'].unique().tolist()}")
print("Nettoyage de MARRIAGE terminé")

# Nettoyage de la colonne EDUCATION
print("\n=== NETTOYAGE DE LA COLONNE EDUCATION ===")
print(f"Valeurs originales dans EDUCATION : {df['EDUCATION'].unique().tolist()}")

# Remplacer les valeurs en dehors de [1, 2, 3, 4] par 4 (autres)
df['EDUCATION'] = df['EDUCATION'].replace([x for x in df['EDUCATION'].unique() if x not in [1, 2, 3, 4]], 4)

print(f"Valeurs après nettoyage : {df['EDUCATION'].unique().tolist()}")
print("Nettoyage de EDUCATION terminé")

# Afficher les répartitions après nettoyage
print("\n=== RÉPARTITION APRÈS NETTOYAGE ===")
print("MARRIAGE :")
marriage_counts = df['MARRIAGE'].value_counts().sort_index()
for value, count in marriage_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")

print("\nEDUCATION :")
education_counts = df['EDUCATION'].value_counts().sort_index()
for value, count in education_counts.items():
    percentage = (count / len(df)) * 100
    print(f"  - Valeur {value} : {count} sur {len(df)} ({percentage:.2f}%)")


=== RÉPARTITION AVANT NETTOYAGE ===
MARRIAGE :
  - Valeur 0 : 54 sur 30000 (0.18%)
  - Valeur 1 : 13659 sur 30000 (45.53%)
  - Valeur 2 : 15964 sur 30000 (53.21%)
  - Valeur 3 : 323 sur 30000 (1.08%)

EDUCATION :
  - Valeur 0 : 14 sur 30000 (0.05%)
  - Valeur 1 : 10585 sur 30000 (35.28%)
  - Valeur 2 : 14030 sur 30000 (46.77%)
  - Valeur 3 : 4917 sur 30000 (16.39%)
  - Valeur 4 : 123 sur 30000 (0.41%)
  - Valeur 5 : 280 sur 30000 (0.93%)
  - Valeur 6 : 51 sur 30000 (0.17%)
=== NETTOYAGE DE LA COLONNE MARRIAGE ===
Valeurs originales dans MARRIAGE : [1, 2, 3, 0]
Valeurs après nettoyage : [1, 2, 3]
Nettoyage de MARRIAGE terminé

=== NETTOYAGE DE LA COLONNE EDUCATION ===
Valeurs originales dans EDUCATION : [2, 1, 3, 5, 4, 6, 0]
Valeurs après nettoyage : [2, 1, 3, 4]
Nettoyage de EDUCATION terminé

=== RÉPARTITION APRÈS NETTOYAGE ===
MARRIAGE :
  - Valeur 1 : 13659 sur 30000 (45.53%)
  - Valeur 2 : 15964 sur 30000 (53.21%)
  - Valeur 3 : 377 sur 30000 (1.26%)

EDUCATION :
  - Valeur 1 : 10

In [3]:
import os
from pathlib import Path

def save_cleaned_dataset(df, filename='cleaned_creditcard.csv'):
    """
    Sauvegarde le DataFrame nettoyé dans le dossier data à la racine
    
    Parameters:
    df (pd.DataFrame): DataFrame nettoyé à sauvegarder
    filename (str): Nom du fichier de sortie
    """
    
    # Construire le chemin vers le dossier data à la racine
    # On utilise Path pour une gestion plus robuste des chemins
    current_path = Path.cwd()  # Chemin du dossier courant
    root_dir = current_path.parent  # Dossier racine (un niveau au-dessus de src)
    data_dir = root_dir / 'data'  # Dossier data
    
    # Vérifier que le dossier data existe, sinon le créer
    if not data_dir.exists():
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"Dossier {data_dir} créé")
    
    # Construire le chemin complet du fichier
    file_path = data_dir / filename
    
    # Sauvegarder le DataFrame en CSV
    df.to_csv(file_path, index=False, encoding="utf-8")
    
    print(f"DataFrame sauvegardé dans : {file_path}")
    print(f"Nombre de lignes : {len(df)}")
    print(f"Nombre de colonnes : {len(df.columns)}")

In [4]:
# enregistrement du CSV pret pour ingestion dans la BDD
save_cleaned_dataset(df=df, filename = 'creditcard_pret_ingestion.csv')

DataFrame sauvegardé dans : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\creditcard_pret_ingestion.csv
Nombre de lignes : 30000
Nombre de colonnes : 25


## Nettoyages décidés après EDA_lab

**Corrections niveau 1**  
- 4 lignes avec PAY_AMTn > 1 000 000
- 860 comptes inactifs : PAY_AMTn == 0 et BILL_AMTn <= 0 sur les 6 mois
- le client 6783 a une codification PAY = 1 sur 4 mois alors qu'il paie chaque mois -> remettre sa codification en 0

**Corrections niveau 2**  
*Correction des PAY_n = 1*  
- Correction des PAY_n = 1 : PAY_n = PAY_(n+1) si BILL_AMT(n+1) <= 0
- correction des PAY_n = 1 restants : PAY_n = 0 si BILL-AMT(n+1) <= 0
- si PAY_2 <=0 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2 
- si PAY_2 = 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 10 alors PAY_1 = 0
- si PAY_2 > 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2
Si ratio_PAY_AMT1_to_BILL_AMT2 < 4 alors il est impossible d'effectuer avec certitude de correction, le client peut continuer à etre en retard, le client peut avoir payer après le batch des 30 jours ou ne pas avoir payé du tout :
- je maitiens PAY_1 = 1 si PAY_2 < 2
- PAY_1 = PAY_2 si PAY_2 >= 2 pour considérer que le client n'a pas résorbé sa dette et que le retard est maintenu

In [5]:
bill_cols = [f'BILL_AMT{i}' for i in range(1, 7)]
pay_cols = [f'PAY_AMT{i}' for i in range(1, 7)]

# 1. Filtre sur les paiements extrêmes (> 1 000 000)
mask_extreme_pay = (df[pay_cols] > 1000000).any(axis=1)

# 2. Filtre sur les inactifs (BILL == 0 et PAY == 0)
mask_inactif = (df[bill_cols] <= 0).all(axis=1) & (df[pay_cols] == 0).all(axis=1)

# Affichage du détail par catégorie
print("=== DÉTAIL DES SUPPRESSIONS ===")
print(f"1. Paiements extrêmes (> 1 000 000)                     : {mask_extreme_pay.sum()}")
print(f"2. Inactifs (BILL <= 0 & PAY = 0)                 : {mask_inactif.sum()}")


# Agrégation des règles d'exclusion
mask_to_remove = mask_extreme_pay | mask_inactif

# Application du nettoyage
df_clean = df[~mask_to_remove].copy()

print("\n=== BILAN GLOBAL ===")
print(f"Total supprimé    : {mask_to_remove.sum()} lignes ({mask_to_remove.mean():.2%})")
print(f"Lignes conservées : {len(df_clean)} / {len(df)}")

=== DÉTAIL DES SUPPRESSIONS ===
1. Paiements extrêmes (> 1 000 000)                     : 4
2. Inactifs (BILL <= 0 & PAY = 0)                 : 860

=== BILAN GLOBAL ===
Total supprimé    : 864 lignes (2.88%)
Lignes conservées : 29136 / 30000


In [6]:
# Vérifier les valeurs actuelles avant modification
print("Valeurs avant modification :")
ligne_6783 = df_clean[df_clean['ID'] == 6783]
print(ligne_6783[['ID','PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5']])

# Modifier les colonnes PAY_1 à PAY_4 de 1 à 0 pour l'ID 6783
df_clean.loc[df['ID'] == 6783, ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']] = 0

# Vérifier les nouvelles valeurs
print("\nValeurs après modification :")
ligne_6783_apres = df_clean[df_clean['ID'] == 6783]
print(ligne_6783_apres[['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4']])

print("\nModification appliquée avec succès !")

Valeurs avant modification :
        ID  PAY_1  PAY_2  PAY_3  PAY_4  PAY_5
6782  6783      1      1      1      1      0

Valeurs après modification :
      PAY_1  PAY_2  PAY_3  PAY_4
6782      0      0      0      0

Modification appliquée avec succès !


In [7]:
save_cleaned_dataset(df_clean, 'cleaned1_creditcard.csv')

DataFrame sauvegardé dans : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\cleaned1_creditcard.csv
Nombre de lignes : 29136
Nombre de colonnes : 25


**Corrections niveau 2**

In [8]:
# Compter le nombre de 1 dans la colonne PAY_1
compte_1 = (df_clean['PAY_1'] == 1).sum()

# Calculer le pourcentage
total = len(df_clean)
pourcentage_1 = (compte_1 / total) * 100

print(f"Nombre de 1 dans la colonne PAY_1 : {compte_1}")
print(f"Pourcentage de 1 dans la colonne PAY_1 : {pourcentage_1:.2f}%")

Nombre de 1 dans la colonne PAY_1 : 3147
Pourcentage de 1 dans la colonne PAY_1 : 10.80%


In [9]:
# Correction de PAY_n = PAY_(n+1) si (BILL_AMT(n+1) <= 0 et PAY_n = 1)
# En partant de PAY_5 vers PAY_1

nb_corrections = 0

for i in range(5, 0, -1):  # De PAY_5 à PAY_1
    col_pay = f'PAY_{i}'
    col_pay_suivant = f'PAY_{i+1}'
    col_bill = f'BILL_AMT{i+1}'
    
    if col_pay in df_clean.columns and col_pay_suivant in df_clean.columns and col_bill in df_clean.columns:
        # Condition : BILL_AMT(n+1) <= 0 et PAY_n = 1
        condition = (df_clean[col_bill] <= 0) & (df_clean[col_pay] == 1)
        
        # Compter les corrections
        nb_corr = condition.sum()
        nb_corrections += nb_corr
        
        if nb_corr > 0:
            print(f"Correction appliquée sur {nb_corr} lignes : {col_pay} devient {col_pay_suivant}")
            # Appliquer la correction
            df_clean.loc[condition, col_pay] = df_clean.loc[condition, col_pay_suivant]

print(f"Nombre total de corrections appliquées : {nb_corrections}")

Correction appliquée sur 12 lignes : PAY_2 devient PAY_3
Correction appliquée sur 635 lignes : PAY_1 devient PAY_2
Nombre total de corrections appliquées : 647


In [10]:
# Compter le nombre total de corrections à appliquer
nb_total_corrections = 0

print("Analyse et corrections :")

# Parcourir chaque mois PAY_1 à PAY_5
for i in range(1, 6):
    col_pay = f'PAY_{i}'
    col_bill = f'BILL_AMT{i+1}'
    
    if col_pay in df_clean.columns and col_bill in df_clean.columns:
        # Condition : PAY_n = 1 et BILL_AMT(n+1) <= 0
        condition = (df_clean[col_pay] == 1) & (df_clean[col_bill] <= 0)
        
        # Compter les corrections à appliquer
        nb_corrections = condition.sum()
        nb_total_corrections += nb_corrections
        
        print(f"PAY_{i} = 1 avec {col_bill} <= 0 : {nb_corrections} corrections")
        
        # Appliquer la correction si nécessaire
        if nb_corrections > 0:
            df_clean.loc[condition, col_pay] = 0

print(f"\nNombre total de corrections appliquées : {nb_total_corrections}")
print("Corrections appliquées avec succès !")

# Vérification : Afficher les résultats après correction
print("\nVérification des éventuels résidus :")
for i in range(1, 6):
    col_pay = f'PAY_{i}'
    col_bill = f'BILL_AMT{i+1}'
    
    if col_pay in df_clean.columns and col_bill in df_clean.columns:
        # Compter les clients avec PAY_n = 1 et BILL_AMT(n+1) <= 0 (devrait être 0 maintenant)
        condition = (df_clean[col_pay] == 1) & (df_clean[col_bill] <= 0)
        nb_restant = condition.sum()
        
        if nb_restant > 0:
            print(f"PAY_{i} = 1 avec {col_bill} <= 0 : {nb_restant} lignes restantes")
        else :
            print(f'pas de résidu sur PAY_{i}')

Analyse et corrections :
PAY_1 = 1 avec BILL_AMT2 <= 0 : 9 corrections
PAY_2 = 1 avec BILL_AMT3 <= 0 : 0 corrections
PAY_3 = 1 avec BILL_AMT4 <= 0 : 0 corrections
PAY_4 = 1 avec BILL_AMT5 <= 0 : 0 corrections
PAY_5 = 1 avec BILL_AMT6 <= 0 : 0 corrections

Nombre total de corrections appliquées : 9
Corrections appliquées avec succès !

Vérification des éventuels résidus :
pas de résidu sur PAY_1
pas de résidu sur PAY_2
pas de résidu sur PAY_3
pas de résidu sur PAY_4
pas de résidu sur PAY_5


- si PAY_2 <=0 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2 
- si PAY_2 = 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 10 alors PAY_1 = 0
- si PAY_2 > 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2
Si ratio_PAY_AMT1_to_BILL_AMT2 < 4 alors il est impossible d'effectuer avec certitude de correction, le client peut continuer à etre en retard, le client peut avoir payer après le batch des 30 jours ou ne pas avoir payé du tout :
- je maitiens PAY_1 = 1 si PAY_2 < 2
- PAY_1 = PAY_2 si PAY_2 >= 2 pour considérer que le client n'a pas résorbé sa dette et que le retard est maintenu

Je sépare ce nettoyage des précédents avec un nouveau dataset à la clé 'cleaned2_creditcard.csv'
J'ai besoin de créer les colonnes ratio_PAY_to_BILL pour poursuivre le nettoyage

In [11]:
import pandas as pd
import numpy as np

# Définition des colonnes
payment_columns = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5']
bill_columns = ['BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']

print("Calcul des ratios conditionnels (uniquement si BILL > 0) :\n")

for i, (pay_col, bill_col) in enumerate(zip(payment_columns, bill_columns)):
    new_col_name = f'ratio_{pay_col}_to_{bill_col}'
    
    # Création de la nouvelle colonne initialisée à NaN
    df_clean[new_col_name] = np.nan
    
    # Sélection des lignes où la facture (dénominateur) est strictement positive
    valid_mask = df_clean[bill_col] > 0
    
    if valid_mask.any():
        # Calcul du ratio uniquement sur les lignes valides
        # On évite ainsi la division par zéro ou par un négatif
        df_clean.loc[valid_mask, new_col_name] = (df_clean.loc[valid_mask, pay_col] / df_clean.loc[valid_mask, bill_col]) * 100
        
        print(f"- {new_col_name}: Calculé sur {valid_mask.sum()} lignes valides")
    else:
        print(f"- {new_col_name}: Aucune facture positive trouvée ({bill_col} <= 0 pour tous), colonne vide (NaN).")

# Affichage des nouvelles colonnes créées
print("\nNouvelles colonnes créées :")
for i, (pay_col, bill_col) in enumerate(zip(payment_columns, bill_columns)):
    new_col_name = f'ratio_{pay_col}_to_{bill_col}'
    print(f"- {new_col_name}")

# Vérification rapide pour voir combien de NaN il y a dans une colonne exemple
example_col = 'ratio_PAY_AMT1_to_BILL_AMT2'
print(f"\nVérification sur '{example_col}':")
print(f" - Valeurs valides (non-NaN) : {df_clean[example_col].notna().sum()}")
print(f" - Valeurs NaN : {df_clean[example_col].isna().sum()}")

Calcul des ratios conditionnels (uniquement si BILL > 0) :

- ratio_PAY_AMT1_to_BILL_AMT2: Calculé sur 26823 lignes valides
- ratio_PAY_AMT2_to_BILL_AMT3: Calculé sur 26471 lignes valides
- ratio_PAY_AMT3_to_BILL_AMT4: Calculé sur 26126 lignes valides
- ratio_PAY_AMT4_to_BILL_AMT5: Calculé sur 25835 lignes valides
- ratio_PAY_AMT5_to_BILL_AMT6: Calculé sur 25288 lignes valides

Nouvelles colonnes créées :
- ratio_PAY_AMT1_to_BILL_AMT2
- ratio_PAY_AMT2_to_BILL_AMT3
- ratio_PAY_AMT3_to_BILL_AMT4
- ratio_PAY_AMT4_to_BILL_AMT5
- ratio_PAY_AMT5_to_BILL_AMT6

Vérification sur 'ratio_PAY_AMT1_to_BILL_AMT2':
 - Valeurs valides (non-NaN) : 26823
 - Valeurs NaN : 2313


In [12]:
# État des lieux initial
print("=== ÉTAT DES LIEUX AVANT CORRECTIONS ===")
nb_pay1_1_initial = (df_clean['PAY_1'] == 1).sum()
pourcentage_initial = (nb_pay1_1_initial / len(df_clean)) * 100
print(f"Nombre de clients avec PAY_1 = 1 : {nb_pay1_1_initial}")
print(f"Pourcentage de clients avec PAY_1 = 1 : {pourcentage_initial:.2f}%")
print(f"Total lignes dans le dataset : {len(df_clean)}")

# Correction 1 : si PAY_2 <= 0 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2
# Mais seulement sur les lignes où PAY_1 = 1 initialement
print("\n=== CORRECTION 1 ===")
condition1_initiale = df_clean['PAY_1'] == 1  # Seulement les lignes avec PAY_1 = 1
condition1 = condition1_initiale & (df_clean['PAY_2'] <= 0) & (df_clean['ratio_PAY_AMT1_to_BILL_AMT2'] > 4)
nb_corr1 = condition1.sum()
print(f"Lignes concernées : {nb_corr1}")
if nb_corr1 > 0:
    df_clean.loc[condition1, 'PAY_1'] = df_clean.loc[condition1, 'PAY_2']
print(f"Corrections appliquées : {nb_corr1}")

# Correction 2 : si PAY_2 = 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 10 alors PAY_1 = 0
print("\n=== CORRECTION 2 ===")
condition2_initiale = df_clean['PAY_1'] == 1
condition2 = condition2_initiale & (df_clean['PAY_2'] == 2) & (df_clean['ratio_PAY_AMT1_to_BILL_AMT2'] > 10)
nb_corr2 = condition2.sum()
print(f"Lignes concernées : {nb_corr2}")
if nb_corr2 > 0:
    df_clean.loc[condition2, 'PAY_1'] = 0
print(f"Corrections appliquées : {nb_corr2}")

# Correction 3 : si PAY_2 > 2 et ratio_PAY_AMT1_to_BILL_AMT2 > 4 alors PAY_1 = PAY_2
print("\n=== CORRECTION 3 ===")
condition3_initiale = df_clean['PAY_1'] == 1
condition3 = condition3_initiale & (df_clean['PAY_2'] > 2) & (df_clean['ratio_PAY_AMT1_to_BILL_AMT2'] > 4)
nb_corr3 = condition3.sum()
print(f"Lignes concernées : {nb_corr3}")
if nb_corr3 > 0:
    df_clean.loc[condition3, 'PAY_1'] = df_clean.loc[condition3, 'PAY_2']
print(f"Corrections appliquées : {nb_corr3}")

# Correction 4 : Si ratio_PAY_AMT1_to_BILL_AMT2 < 4
print("\n=== CORRECTION 4 ===")
condition4_initiale = df_clean['PAY_1'] == 1
condition4 = condition4_initiale & (df_clean['ratio_PAY_AMT1_to_BILL_AMT2'] < 4)
nb_corr4 = 0

# Pour les lignes où ratio < 4 et PAY_2 >= 2 : PAY_1 = PAY_2
condition4_a = condition4 & (df_clean['PAY_2'] >= 2)
nb_corr4_a = condition4_a.sum()
print(f"Lignes concernées (ratio < 4 et PAY_2 >= 2) : {nb_corr4_a}")
if nb_corr4_a > 0:
    df_clean.loc[condition4_a, 'PAY_1'] = df_clean.loc[condition4_a, 'PAY_2']
    nb_corr4 += nb_corr4_a

print(f"Corrections appliquées : {nb_corr4}")

# État des lieux final
print("\n=== ÉTAT DES LIEUX APRÈS CORRECTIONS ===")
nb_pay1_1_final = (df_clean['PAY_1'] == 1).sum()
pourcentage_final = (nb_pay1_1_final / len(df_clean)) * 100
print(f"Nombre de clients avec PAY_1 = 1 : {nb_pay1_1_final}")
print(f"Pourcentage de clients avec PAY_1 = 1 : {pourcentage_final:.2f}%")

# Résumé des corrections
total_corrections = nb_corr1 + nb_corr2 + nb_corr3 + nb_corr4
print(f"\n=== RÉSUMÉ DES CORRECTIONS ===")
print(f"Total corrections appliquées : {total_corrections}")
print(f"Pourcentage du dataset corrigé : {(total_corrections / len(df_clean)) * 100:.2f}%")

=== ÉTAT DES LIEUX AVANT CORRECTIONS ===
Nombre de clients avec PAY_1 = 1 : 2512
Pourcentage de clients avec PAY_1 = 1 : 8.62%
Total lignes dans le dataset : 29136

=== CORRECTION 1 ===
Lignes concernées : 680
Corrections appliquées : 680

=== CORRECTION 2 ===
Lignes concernées : 160
Corrections appliquées : 160

=== CORRECTION 3 ===
Lignes concernées : 22
Corrections appliquées : 22

=== CORRECTION 4 ===
Lignes concernées (ratio < 4 et PAY_2 >= 2) : 1281
Corrections appliquées : 1281

=== ÉTAT DES LIEUX APRÈS CORRECTIONS ===
Nombre de clients avec PAY_1 = 1 : 369
Pourcentage de clients avec PAY_1 = 1 : 1.27%

=== RÉSUMÉ DES CORRECTIONS ===
Total corrections appliquées : 2143
Pourcentage du dataset corrigé : 7.36%


### Je dois corriger les valeurs aberrantes des colonnes ratio utilisées précédemment
Je les conserve car elles seront utiles pour l'EDA et le ML
Mais certaines valeurs dépassent les 1000% à cause d'un comportement de l'époque et de certaines petites valeurs dues
Correction appliquée :  
écrêtage des valeurs à 200% maximum
création d'une valeur de ratio à partir d'un montant dû de 100 NT$


In [13]:
import pandas as pd
import numpy as np

# Définition des colonnes
payment_columns = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5']
bill_columns = ['BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']

print("Calcul des ratios conditionnels (uniquement si BILL > 100) :\n")

for i, (pay_col, bill_col) in enumerate(zip(payment_columns, bill_columns)):
    new_col_name = f'ratio_{pay_col}_to_{bill_col}'
    
    # Création de la nouvelle colonne initialisée à NaN
    df_clean[new_col_name] = np.nan
    
    # Sélection des lignes où la facture (dénominateur) est strictement positive
    valid_mask = df_clean[bill_col] > 100
    
    if valid_mask.any():
        # Calcul du ratio uniquement sur les lignes valides
        # On évite ainsi la division par zéro ou par un négatif
        ratios = (df_clean.loc[valid_mask, pay_col] / df_clean.loc[valid_mask, bill_col]) * 100
        
        # Limiter la valeur maximale à 200%
        ratios = ratios.clip(upper=200)
        
        df_clean.loc[valid_mask, new_col_name] = ratios
        
        print(f"- {new_col_name}: Calculé sur {valid_mask.sum()} lignes valides")
    else:
        print(f"- {new_col_name}: Aucune facture positive trouvée ({bill_col} <= 0 pour tous), colonne vide (NaN).")

# Affichage des nouvelles colonnes créées
print("\nNouvelles colonnes créées :")
for i, (pay_col, bill_col) in enumerate(zip(payment_columns, bill_columns)):
    new_col_name = f'ratio_{pay_col}_to_{bill_col}'
    print(f"- {new_col_name}")

# Vérification rapide pour voir combien de NaN il y a dans une colonne exemple
example_col = 'ratio_PAY_AMT1_to_BILL_AMT2'
print(f"\nVérification sur '{example_col}':")
print(f" - Valeurs valides (non-NaN) : {df_clean[example_col].notna().sum()}")
print(f" - Valeurs NaN : {df_clean[example_col].isna().sum()}")

Calcul des ratios conditionnels (uniquement si BILL > 100) :

- ratio_PAY_AMT1_to_BILL_AMT2: Calculé sur 26749 lignes valides
- ratio_PAY_AMT2_to_BILL_AMT3: Calculé sur 26401 lignes valides
- ratio_PAY_AMT3_to_BILL_AMT4: Calculé sur 26063 lignes valides
- ratio_PAY_AMT4_to_BILL_AMT5: Calculé sur 25750 lignes valides
- ratio_PAY_AMT5_to_BILL_AMT6: Calculé sur 25215 lignes valides

Nouvelles colonnes créées :
- ratio_PAY_AMT1_to_BILL_AMT2
- ratio_PAY_AMT2_to_BILL_AMT3
- ratio_PAY_AMT3_to_BILL_AMT4
- ratio_PAY_AMT4_to_BILL_AMT5
- ratio_PAY_AMT5_to_BILL_AMT6

Vérification sur 'ratio_PAY_AMT1_to_BILL_AMT2':
 - Valeurs valides (non-NaN) : 26749
 - Valeurs NaN : 2387


In [14]:
df_clean.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,dpnm,ratio_PAY_AMT1_to_BILL_AMT2,ratio_PAY_AMT2_to_BILL_AMT3,ratio_PAY_AMT3_to_BILL_AMT4,ratio_PAY_AMT4_to_BILL_AMT5,ratio_PAY_AMT5_to_BILL_AMT6
count,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,...,29136.000000,29136.000000,29136.000000,29136.000000,29136.000000,26749.000000,26401.000000,26063.000000,25750.000000,25215.000000
mean,15008.506555,166096.570566,1.603034,1.847131,1.559308,35.448826,-0.080759,-0.079833,-0.111958,-0.168040,...,5340.278762,4944.484281,4923.516646,5363.605093,0.216879,34.068994,34.279400,32.442732,31.584128,33.262684
std,8652.776807,129791.076684,0.489277,0.742888,0.521365,9.213621,1.213999,1.170783,1.171552,1.144978,...,17036.118539,15449.165540,15308.183132,17997.105905,0.412127,43.437649,43.776411,43.665768,43.633582,44.094939
min,1.000000,10000.000000,1.000000,1.000000,1.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7525.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,544.750000,390.000000,380.000000,300.000000,0.000000,4.281604,4.240447,3.635189,3.578829,3.676002
50%,15018.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,2000.000000,1600.000000,1641.500000,1592.000000,0.000000,7.802625,7.726184,6.161701,5.160547,5.576001
75%,22487.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,4767.250000,4200.000000,4220.250000,4179.250000,0.000000,100.000000,100.000000,93.870713,78.726592,100.000000
max,30000.000000,1000000.000000,2.000000,4.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,896040.000000,528897.000000,426529.000000,528666.000000,1.000000,200.000000,200.000000,200.000000,200.000000,200.000000


In [15]:
save_cleaned_dataset(df_clean, 'cleaned2_creditcard.csv')

DataFrame sauvegardé dans : c:\Users\johan\VS Code Wild 2\PROJET_DEFAULT_CREDIT_CARD_ML\data\cleaned2_creditcard.csv
Nombre de lignes : 29136
Nombre de colonnes : 30
